In [33]:
import os, sys, json, time, textwrap, importlib, pathlib, torch

ROOT = "/mnt/data/multimodal_low_compute_toolkit"
os.makedirs(ROOT, exist_ok=True)
sys.path.append(ROOT)

def _write(path, content):
    with open(path, "w") as f: f.write(textwrap.dedent(content))

need_files = {"metrics_hooks.py","fusion_mbt.py","freeze_utils.py","model_skeleton.py","run_ablation.py"}
if not need_files.issubset(set(os.listdir(ROOT))):
    # --- metrics_hooks.py ---
    _write(os.path.join(ROOT,"metrics_hooks.py"), r'''
import json, time, torch
from typing import Dict
def _center(x): return x - x.mean(dim=0, keepdim=True)
@torch.no_grad()
def corr_offdiag_stats(E: torch.Tensor) -> Dict[str, float]:
    E = _center(E); E = E / (E.std(dim=0, keepdim=True) + 1e-6)
    C = (E.T @ E) / (E.size(0) - 1)
    off = C - torch.eye(C.size(0), device=C.device)
    offdiag = off[~torch.eye(C.size(0), dtype=bool, device=C.device)]
    return {"offdiag_abs_mean": offdiag.abs().mean().item(), "offdiag_abs_max": offdiag.abs().max().item()}
@torch.no_grad()
def spectral_stats(E: torch.Tensor, ks=(32,64,128)) -> Dict[str,float]:
    E = _center(E); C = (E.T @ E) / (E.size(0) - 1)
    evals = torch.linalg.eigvalsh(C).clamp_min(0); s = evals.sum()
    pr = (s**2 / (evals.pow(2).sum() + 1e-8)).item()
    ev_sorted = torch.sort(evals, descending=True).values
    ev_cum = torch.cumsum(ev_sorted, dim=0) / (ev_sorted.sum() + 1e-8)
    out = {"participation_ratio": pr, "cond_num": (ev_sorted[0] / (ev_sorted[-1] + 1e-8)).item()}
    for k in ks:
        idx = min(k-1, ev_cum.numel()-1); out[f"expl_var@{k}"] = ev_cum[idx].item()
    return out
@torch.no_grad()
def vicreg_collapse(E: torch.Tensor, target_std=1.0) -> Dict[str,float]:
    s = E.std(dim=0); return {"num_collapsed_dims": int((s < target_std).sum().item()), "mean_std": s.mean().item()}
@torch.no_grad()
def topk_dim_dup(E: torch.Tensor, k=3) -> Dict[str,float]:
    E = _center(E); E = E / (E.std(dim=0, keepdim=True) + 1e-6)
    C = (E.T @ E) / (E.size(0) - 1); D = C.size(0); C = C.clone(); C.fill_diagonal_(0.0)
    vals, _ = torch.topk(C.abs(), k=min(k, D-1), dim=1); return {"avg_topk_dim_corr": vals.mean().item()}
class RedundancyProbe:
    def all_stats(self, E: torch.Tensor):
        out = {}; out.update(corr_offdiag_stats(E)); out.update(spectral_stats(E))
        out.update(vicreg_collapse(E)); out.update(topk_dim_dup(E)); return out
@torch.no_grad()
def shared_private_alignment(zs: torch.Tensor, zp: torch.Tensor):
    assert zs.size(1) == zp.size(1), "Dims must match for cosine; pad/truncate first."
    zs_n = zs / (zs.norm(dim=1, keepdim=True) + 1e-6)
    zp_n = zp / (zp.norm(dim=1, keepdim=True) + 1e-6)
    cos = (zs_n * zp_n).sum(dim=1)
    return {"mean_sp_cos": cos.mean().item(), "p95_sp_cos": cos.quantile(0.95).item()}
@torch.no_grad()
def cross_cov_fro(zs: torch.Tensor, zp: torch.Tensor):
    # Allows mismatched dims by padding smaller to larger with zeros.
    D = max(zs.size(1), zp.size(1))
    def pad(x): 
        if x.size(1)==D: return x
        pad_cols = torch.zeros(x.size(0), D-x.size(1), device=x.device, dtype=x.dtype)
        return torch.cat([x, pad_cols], dim=1)
    zs = pad(zs); zp = pad(zp)
    zs_c = (zs - zs.mean(0)) / (zs.std(0)+1e-6)
    zp_c = (zp - zp.mean(0)) / (zp.std(0)+1e-6)
    C = (zs_c.T @ zp_c) / (zs.size(0)-1)
    return {"cross_cov_fro": torch.linalg.matrix_norm(C, ord='fro').item()}
class OrthoProbe:
    def sp_stats(self, zs, zp):
        out = {}; 
        try:
            if zs.size(1)==zp.size(1): out.update(shared_private_alignment(zs, zp))
        except Exception: pass
        out.update(cross_cov_fro(zs, zp)); return out
def orthogonality_loss(zs, zp, lambda_ortho: float = 0.1):
    # Pad to common dim for cross-cov loss too
    D = max(zs.size(1), zp.size(1))
    def pad(x):
        if x.size(1)==D: return x
        pad_cols = torch.zeros(x.size(0), D-x.size(1), device=x.device, dtype=x.dtype)
        return torch.cat([x, pad_cols], dim=1)
    zs = pad(zs); zp = pad(zp)
    zs_c = (zs - zs.mean(0)) / (zs.std(0)+1e-6)
    zp_c = (zp - zp.mean(0)) / (zp.std(0)+1e-6)
    C = (zs_c.T @ zp_c) / (zs.size(0)-1)
    return lambda_ortho * torch.sum(C**2)
def log_vram_and_step_time(start_time_s: float):
    iter_time_ms = (time.time() - start_time_s) * 1000.0
    try: vram_mb = torch.cuda.max_memory_allocated() / (1024**2)
    except Exception: vram_mb = 0.0
    return vram_mb, iter_time_ms
''')
    # --- fusion_mbt.py ---
    _write(os.path.join(ROOT,"fusion_mbt.py"), r'''
import torch, torch.nn as nn
class CrossAttentionBlock(nn.Module):
    def __init__(self, dim=256, heads=4, mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        self.norm_q = nn.LayerNorm(dim); self.norm_kv = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(embed_dim=dim, num_heads=heads, batch_first=True, dropout=dropout)
        self.mlp = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, int(dim*mlp_ratio)), nn.GELU(), nn.Linear(int(dim*mlp_ratio), dim))
    def forward(self, latents, tokens):
        q = self.norm_q(latents); kv = self.norm_kv(tokens)
        out,_ = self.attn(q, kv, kv, need_weights=False)
        latents = latents + out; latents = latents + self.mlp(latents); return latents
class MBTFusion(nn.Module):
    def __init__(self, dim=256, num_latents=4, num_layers=2, heads=4, mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        self.latents = nn.Parameter(torch.randn(1, num_latents, dim) * 0.02)
        self.blocks = nn.ModuleList([CrossAttentionBlock(dim=dim, heads=heads, mlp_ratio=mlp_ratio, dropout=dropout) for _ in range(num_layers)])
        self.norm = nn.LayerNorm(dim)
    def forward(self, tokens_concat: torch.Tensor) -> torch.Tensor:
        B = tokens_concat.size(0); lat = self.latents.expand(B, -1, -1)
        for blk in self.blocks: lat = blk(lat, tokens_concat)
        return self.norm(lat).mean(dim=1)  # pooled latent
''')
    # --- freeze_utils.py ---
    _write(os.path.join(ROOT,"freeze_utils.py"), r'''
import torch.nn as nn
def freeze_all(module: nn.Module):
    for p in module.parameters(): p.requires_grad = False
def count_trainable_params(module: nn.Module) -> int:
    return sum(p.numel() for p in module.parameters() if p.requires_grad)
def unfreeze_last_k_transformer_blocks(hf_transformer: nn.Module, k: int = 1):
    layers = None
    for attr in ["encoder","transformer","vision_model","text_model"]:
        if hasattr(hf_transformer, attr):
            sub = getattr(hf_transformer, attr)
            if hasattr(sub,"layer"): layers = sub.layer; break
            if hasattr(sub,"layers"): layers = sub.layers; break
    if not layers or k<=0: return
    for blk in layers[-k:]:
        for p in blk.parameters(): p.requires_grad = True
def enable_bitfit(module: nn.Module, layernorm_only: bool = True):
    for name, p in module.named_parameters():
        p.requires_grad = False
        if "bias" in name: p.requires_grad = True
        if layernorm_only and any(s in name.lower() for s in ["layernorm","ln"]): p.requires_grad = True
''')
    # --- model_skeleton.py ---
    # --- model_skeleton.py ---
_write(os.path.join(ROOT,"model_skeleton.py"), r'''
import torch
import torch.nn as nn
from fusion_mbt import MBTFusion
from metrics_hooks import orthogonality_loss as _ortho_loss

class ModalityHead(nn.Module):
    """
    Projection head -> (z_shared, z_private)
    - pre LayerNorm on inputs (stabilize encoder scale)
    - LayerNorm on shared output (pins mean/std; helps H1 probes)
    """
    def __init__(self, in_dim, shared=256, private=128, hid=512):
        super().__init__()
        self.pre_ln = nn.LayerNorm(in_dim, elementwise_affine=True)

        self.shared_mlp = nn.Sequential(
            nn.Linear(in_dim, hid), nn.ReLU(),
            nn.Linear(hid, shared)
        )
        self.shared_ln = nn.LayerNorm(shared, elementwise_affine=True)  # NEW

        self.private_mlp = nn.Sequential(
            nn.Linear(in_dim, hid), nn.ReLU(),
            nn.Linear(hid, private)
        )

    def forward(self, x):
        x = self.pre_ln(x)
        z_s = self.shared_mlp(x)
        z_s = self.shared_ln(z_s)   # NEW: stabilize shared scale for decorrelation
        z_p = self.private_mlp(x)
        return z_s, z_p


class MultiModalTiny(nn.Module):
    """
    Tiny multimodal head + MBT fusion:
      - Per-modality heads → 256-D shared & 128-D private
      - MBT fusion over shared tokens
      - Classifier on fused shared token

    in_dims lets you pass CLIP=512, DistilBERT/HuBERT=768 directly.
    """
    def __init__(self,
                 dim=256,
                 num_latents=4,
                 num_layers=2,
                 lambda_ortho=0.1,
                 in_dims=dict(img=768, txt=768, aud=768)):  # NEW
        super().__init__()
        self.lambda_ortho = lambda_ortho

        self.img_head = ModalityHead(in_dim=in_dims["img"], shared=dim, private=128)
        self.txt_head = ModalityHead(in_dim=in_dims["txt"], shared=dim, private=128)
        self.aud_head = ModalityHead(in_dim=in_dims["aud"], shared=dim, private=128)

        self.fusion = MBTFusion(dim=dim, num_latents=num_latents, num_layers=num_layers)
        self.classifier = nn.Linear(dim, 2)

    def forward(self, img_feat, txt_feat, aud_feat, compute_losses=True):
        zsi, zpi = self.img_head(img_feat)
        zst, zpt = self.txt_head(txt_feat)
        zsa, zpa = self.aud_head(aud_feat)

        # stack shared tokens for MBT (N, M=3, D_shared)
        tokens = torch.stack([zsi, zst, zsa], dim=1)
        fused  = self.fusion(tokens)        # (N, D_shared)
        logits = self.classifier(fused)

        losses = {}
        if compute_losses:
            L_ortho  = _ortho_loss(zsi, zpi, self.lambda_ortho)
            L_ortho += _ortho_loss(zst, zpt, self.lambda_ortho)
            L_ortho += _ortho_loss(zsa, zpa, self.lambda_ortho)
            losses["L_ortho"] = L_ortho

        return logits, dict(zsi=zsi, zpi=zpi, zst=zst, zpt=zpt, zsa=zsa, zpa=zpa), losses
''')


print("Toolkit ready at", ROOT)
print("Files:", os.listdir(ROOT))


Toolkit ready at /mnt/data/multimodal_low_compute_toolkit
Files: ['model_skeleton.py', 'fusion_mbt.py', '__pycache__', 'metrics_hooks.py', 'freeze_utils.py', 'h3_ablation.json']


In [2]:
import importlib, json, time, torch
metrics_hooks = importlib.import_module("metrics_hooks")
model_skeleton = importlib.import_module("model_skeleton")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

def synthetic_batch(bs=64, din=768, device="cpu"):
    return (torch.randn(bs, din, device=device),
            torch.randn(bs, din, device=device),
            torch.randn(bs, din, device=device))

def run_h3_ablation(mbt_tokens=(0,4,6,8), layers=2, dim=256, save_json=None):
    rows = []
    for t in mbt_tokens:
        model = model_skeleton.MultiModalTiny(dim=dim, num_latents=t, num_layers=layers).to(device)
        params_m = sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6
        img, txt, aud = synthetic_batch(device=device)
        start = time.time()
        logits, zs, losses = model(img, txt, aud, compute_losses=True)
        vram_mb, iter_ms = metrics_hooks.log_vram_and_step_time(start)
        metric_val = float(torch.softmax(logits, dim=1).max(dim=1).values.mean().item())
        rows.append(dict(run=f"MBT{t}", vram_mb=vram_mb, iter_ms=iter_ms, metric=metric_val, params_m=params_m))
        print(f"{rows[-1]['run']:5s} | VRAM {vram_mb:7.1f} MB | {iter_ms:7.1f} ms | metric {metric_val:.3f} | params {params_m:.2f} M")
    if save_json:
        with open(save_json, "w") as f: json.dump(rows, f, indent=2)
    rows_path = save_json or "<not saved>"
    print("Saved:", rows_path)
    return rows

h3_rows = run_h3_ablation(save_json=os.path.join(ROOT,"h3_ablation.json"))


Device: cuda
MBT0  | VRAM    31.9 MB |   408.2 ms | metric nan | params 4.54 M
MBT4  | VRAM    62.6 MB |    37.7 ms | metric 0.687 | params 4.55 M
MBT6  | VRAM    73.6 MB |     5.8 ms | metric 0.666 | params 4.55 M
MBT8  | VRAM    81.7 MB |     5.0 ms | metric 0.631 | params 4.55 M
Saved: /mnt/data/multimodal_low_compute_toolkit/h3_ablation.json


In [3]:
import importlib, torch
metrics_hooks = importlib.import_module("metrics_hooks")
probe = metrics_hooks.RedundancyProbe()

N, D = 512, 256
# correlated embedding by duplicating features + small noise
base = torch.randn(N, D//4)
E_before = torch.cat([base, base, base, base], dim=1) + 0.05*torch.randn(N, D)

# decorrelated via whitening (rough proxy for Barlow/VICReg effect)
X = E_before - E_before.mean(0, keepdim=True)
C = (X.T @ X) / (N-1)
evals, evecs = torch.linalg.eigh(C + 1e-3*torch.eye(D))
W = evecs @ torch.diag((evals.clamp_min(1e-6)).pow(-0.5)) @ evecs.T
E_after = X @ W  # whitened (unit-cov in expectation)

stats_before = probe.all_stats(E_before)
stats_after  = probe.all_stats(E_after)
print("OFFDIAG mean (↓ better): before={:.3f}  after={:.3f}".format(stats_before['offdiag_abs_mean'], stats_after['offdiag_abs_mean']))
print("Collapsed dims (↓):      before={}      after={}".format(stats_before['num_collapsed_dims'], stats_after['num_collapsed_dims']))
print("PR (↑):                  before={:.1f}  after={:.1f}".format(stats_before['participation_ratio'], stats_after['participation_ratio']))
print("Top-k dim corr (↓):      before={:.3f}  after={:.3f}".format(stats_before['avg_topk_dim_corr'], stats_after['avg_topk_dim_corr']))

stats_before, stats_after


OFFDIAG mean (↓ better): before=0.046  after=0.012
Collapsed dims (↓):      before=127      after=256
PR (↑):                  before=57.0  after=233.6
Top-k dim corr (↓):      before=0.998  after=0.135


({'offdiag_abs_mean': 0.046494171023368835,
  'offdiag_abs_max': 0.997972309589386,
  'participation_ratio': 56.985809326171875,
  'cond_num': 27197.5,
  'expl_var@32': 0.6483073234558105,
  'expl_var@64': 0.9983687996864319,
  'expl_var@128': 0.9993425607681274,
  'num_collapsed_dims': 127,
  'mean_std': 1.0008254051208496,
  'avg_topk_dim_corr': 0.9975059032440186},
 {'offdiag_abs_mean': 0.01166560035198927,
  'offdiag_abs_max': 0.17487840354442596,
  'participation_ratio': 233.58851623535156,
  'cond_num': 4.820189476013184,
  'expl_var@32': 0.17542217671871185,
  'expl_var@64': 0.35081321001052856,
  'expl_var@128': 0.6281422972679138,
  'num_collapsed_dims': 256,
  'mean_std': 0.8440219759941101,
  'avg_topk_dim_corr': 0.13450069725513458})

In [6]:
import importlib, torch
metrics_hooks = importlib.import_module("metrics_hooks")
ortho = metrics_hooks.OrthoProbe()

N = 512
zs = torch.randn(N, 256)
#  zp correlated to zs via a random linear map, then project to 128-D
A = torch.randn(256, 128) / 10.0
zp_corr = zs @ A + 0.1*torch.randn(N, 128)

# 1) Baseline overlap
print("Baseline cross-cov Fro (higher = worse overlap):",
      metrics_hooks.cross_cov_fro(zs, zp_corr)['cross_cov_fro'])

# 2) Center & scale both (like the orthogonality loss does)
zsc = (zs - zs.mean(0)) / (zs.std(0) + 1e-6)    # N x 256
zpc = (zp_corr - zp_corr.mean(0)) / (zp_corr.std(0) + 1e-6)  # N x 128

# 3) Regress zpc on zsc (least squares), then remove the explained part
#  Finds W that minimizes ||zpc - zsc @ W||_F
try:
    W = torch.linalg.lstsq(zsc, zpc).solution         # 256 x 128 (PyTorch ≥ 1.9)
except Exception:
    W = torch.linalg.pinv(zsc) @ zpc                  # fallback

zpc_hat = zsc @ W                                     # N x 128 (part of zpc explained by zs)
zp_orth = zpc - zpc_hat                                # residual: orthogonal component

# 4) Overlap after orthogonalization (should drop)
print("Orthogonalized cross-cov Fro (should drop):",
      metrics_hooks.cross_cov_fro(zs, zp_orth)['cross_cov_fro'])

# 5) Example: orthogonality loss value on original correlated pair
L_ortho_val = metrics_hooks.orthogonality_loss(zs, zp_corr, lambda_ortho=0.1).item()
print("Orthogonality loss (λ=0.1) on correlated pair:", L_ortho_val)


Baseline cross-cov Fro (higher = worse overlap): 13.780684471130371
Orthogonalized cross-cov Fro (should drop): 0.00011768912372644991
Orthogonality loss (λ=0.1) on correlated pair: 18.9907283782959


In [19]:
# freezing / partial unfreezing demo (now with params > 0)
import importlib, torch, torch.nn as nn
freeze_utils = importlib.import_module("freeze_utils")

class DummyBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.lin = nn.Linear(16, 16)
        self.ln  = nn.LayerNorm(16)
    def forward(self, x): return self.ln(self.lin(x))

class DummyEncoder(nn.Module):
    def __init__(self, L=6):
        super().__init__()
        self.encoder = type("E", (), {})()
        self.encoder.layer = nn.ModuleList([DummyBlock() for _ in range(L)])

enc = DummyEncoder(L=6)
print("Total params:", sum(p.numel() for p in enc.parameters()))
freeze_utils.freeze_all(enc)
print("Trainable after freeze:", freeze_utils.count_trainable_params(enc))
freeze_utils.unfreeze_last_k_transformer_blocks(enc, k=2)
print("Trainable after unfreezing last 2 blocks:", freeze_utils.count_trainable_params(enc))
freeze_utils.enable_bitfit(enc, layernorm_only=True)
print("Trainable after BitFit (LN/bias):", freeze_utils.count_trainable_params(enc))


Total params: 0
Trainable after freeze: 0
Trainable after unfreezing last 2 blocks: 0
Trainable after BitFit (LN/bias): 0


In [8]:
import importlib, torch, time
metrics_hooks = importlib.import_module("metrics_hooks")
model_skeleton = importlib.import_module("model_skeleton")

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model_skeleton.MultiModalTiny(dim=256, num_latents=6, num_layers=2, lambda_ortho=0.1).to(device)
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3)

def get_batch(bs=64, din=768):
    # To replace with real dataloader (img/text/audio features)
    return (torch.randn(bs, din, device=device),
            torch.randn(bs, din, device=device),
            torch.randn(bs, din, device=device),
            torch.randint(0,2,(bs,), device=device))

criterion = torch.nn.CrossEntropyLoss()

for step in range(3):  # demo steps
    img, txt, aud, y = get_batch()
    start = time.time()
    logits, zs, losses = model(img, txt, aud, compute_losses=True)
    L_task = criterion(logits, y)
    L_ortho = losses["L_ortho"]
    # Dry run to  add your Barlow/VICReg and contrastive losses here: L_red, L_align
    loss = L_task + L_ortho
    opt.zero_grad(); loss.backward(); opt.step()
    vram_mb, iter_ms = metrics_hooks.log_vram_and_step_time(start)
    print(f"step {step} | loss={loss.item():.3f} | L_task={L_task.item():.3f} | L_ortho={L_ortho.item():.3f} | VRAM={vram_mb:.1f}MB | {iter_ms:.1f}ms")

    # H1 probe (per modality)
    red = metrics_hooks.RedundancyProbe()
    r_shared = red.all_stats(zs["zsi"].detach())
    r_private= red.all_stats(zs["zpi"].detach())
    # H1: log/compare these before vs after enabling Barlow/VICReg

    # Orthogonality probe (PRISM goal)
    ortho = metrics_hooks.OrthoProbe()
    sp_stats = ortho.sp_stats(zs["zsi"].detach(), zs["zpi"].detach())
    # Expect mean_sp_cos ~ 0, cross_cov_fro trending down when λ_ortho > 0

print("Done.")


step 0 | loss=163.025 | L_task=0.726 | L_ortho=162.298 | VRAM=104.3MB | 314.6ms
step 1 | loss=164.042 | L_task=1.572 | L_ortho=162.470 | VRAM=104.3MB | 13.6ms
step 2 | loss=164.263 | L_task=2.729 | L_ortho=161.534 | VRAM=104.3MB | 11.9ms
Done.


In [11]:
from transformers import AutoModel, AutoTokenizer, CLIPProcessor, CLIPModel
from transformers import HubertModel, Wav2Vec2FeatureExtractor
import torch, numpy as np
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"

# IMAGE (CLIP ViT-B/16 via HF) 
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch16").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch16")
clip_model.eval()
for p in clip_model.parameters(): p.requires_grad = False

# TEXT (DistilBERT)
text_encoder = AutoModel.from_pretrained("distilbert-base-uncased").to(device)
text_tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")
text_encoder.eval()
for p in text_encoder.parameters(): p.requires_grad = False

#  AUDIO (HuBERT)
aud_proc = Wav2Vec2FeatureExtractor.from_pretrained("facebook/hubert-base-ls960")
aud_enc  = HubertModel.from_pretrained("facebook/hubert-base-ls960").to(device)
aud_enc.eval()
for p in aud_enc.parameters(): p.requires_grad = False


2025-10-03 13:24:40.595648: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759497880.802220      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759497880.863783      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/599M [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/213 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

In [18]:
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import importlib
import torch.nn.functional as F


@torch.no_grad()
def featurize_images(pils):
    inputs = clip_processor(images=pils, return_tensors="pt").to(device)
    x = clip_model.get_image_features(**inputs)  # [N, 512] for HF CLIP ViT-B/16
    return x.float()

@torch.no_grad()
def featurize_text(texts):
    tok = text_tok(texts, return_tensors="pt", padding=True, truncation=True, max_length=96).to(device)
    out = text_encoder(**tok).last_hidden_state  # [N, T, 768]
    return out.mean(dim=1).float()               # [N, 768]

@torch.no_grad()
def featurize_audio(batch_wav16k):
    proc = aud_proc(batch_wav16k, sampling_rate=16000, return_tensors="pt", padding=True)
    input_values = proc["input_values"].to(device)
    attention_mask = proc.get("attention_mask", None)
    if attention_mask is not None:
        attention_mask = attention_mask.to(device)
    out = aud_enc(input_values=input_values, attention_mask=attention_mask).last_hidden_state  # [N, T, 768]
    if out.size(1) > 96:  # downsample frames to keep compute small
        stride = int(np.ceil(out.size(1) / 96))
        out = out[:, ::stride]
    return out.mean(dim=1).float()  # [N, 768]

# tiny image adapter: 512 -> 768 (so heads can stay at in_dim=768) 
img_adapter = nn.Linear(512, 768).to(device)
with torch.no_grad():
    nn.init.xavier_uniform_(img_adapter.weight)
    nn.init.zeros_(img_adapter.bias)

# model + probes (use a smaller ortho weight at init) 
from model_skeleton import MultiModalTiny
metrics_hooks = importlib.import_module("metrics_hooks")

fusion_dim = 256
model = MultiModalTiny(dim=fusion_dim, num_latents=6, num_layers=2, lambda_ortho=0.01).to(device)
model.eval()

N = 128
imgs  = [Image.fromarray((np.random.rand(224,224,3)*255).astype(np.uint8)) for _ in range(N)]
texts = [f"random text {i}" for i in range(N)]
audios = [np.random.randn(16000).astype(np.float32) for _ in range(N)]

with torch.no_grad():
    img_512 = featurize_images(imgs)      # [N, 512]
    img_768 = img_adapter(img_512)        # [N, 768]
    txt_768 = featurize_text(texts)       # [N, 768]
    aud_768 = featurize_audio(audios)     # [N, 768]

logits, zs, losses = model(img_768, txt_768, aud_768, compute_losses=True)
print("logits:", tuple(logits.shape), "| L_ortho:", float(losses["L_ortho"]))

# H1 redundancy stats 
probe = metrics_hooks.RedundancyProbe()
def show(tag, E):
    s = probe.all_stats(E.detach())
    print(f"{tag:12s} | offdiag={s['offdiag_abs_mean']:.3f} | PR={s['participation_ratio']:.1f} | "
          f"collapsed={s['num_collapsed_dims']} | topk_dup={s['avg_topk_dim_corr']:.3f}")

show("img_shared", zs["zsi"])
show("img_private", zs["zpi"])
show("txt_shared", zs["zst"])
show("txt_private", zs["zpt"])
show("aud_shared", zs["zsa"])
show("aud_private", zs["zpa"])

logits: (128, 2) | L_ortho: 92.62114715576172
img_shared   | offdiag=0.242 | PR=8.2 | collapsed=256 | topk_dup=0.608
img_private  | offdiag=0.241 | PR=6.1 | collapsed=128 | topk_dup=0.590
txt_shared   | offdiag=0.194 | PR=14.6 | collapsed=256 | topk_dup=0.573
txt_private  | offdiag=0.179 | PR=15.6 | collapsed=128 | topk_dup=0.513
aud_shared   | offdiag=0.356 | PR=3.8 | collapsed=256 | topk_dup=0.773
aud_private  | offdiag=0.307 | PR=5.1 | collapsed=128 | topk_dup=0.713


In [34]:
# H1 DECORRELATION: BT + CORR_OFFDIAG + STRONG VAR, WITH LAYERNORMED SHARED HEADS
import torch, numpy as np, random
from PIL import Image
import importlib
metrics_hooks = importlib.import_module("metrics_hooks")
from model_skeleton import MultiModalTiny

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(123); np.random.seed(123); random.seed(123)


def corr_offdiag(E):
    Ez = (E - E.mean(0, keepdim=True)) / (E.std(0, keepdim=True) + 1e-6)
    C  = (Ez.T @ Ez) / (E.size(0) - 1)             # correlation
    off = C - torch.diag(torch.diag(C))
    return (off.pow(2).mean())

def variance_term(E, gamma=1.0, eps=1e-4):
    s = E.std(0) + eps
    return torch.mean(torch.relu(gamma - s).pow(2))

def barlow_twins(z1, z2, lambd=5e-3):
    z1n = (z1 - z1.mean(0, True)) / (z1.std(0, True) + 1e-6)
    z2n = (z2 - z2.mean(0, True)) / (z2.std(0, True) + 1e-6)
    C = (z1n.T @ z2n) / (z1n.size(0) - 1)
    on  = torch.diagonal(C) - 1
    off = C - torch.diag(torch.diag(C))
    return (on.pow(2).sum() + lambd * off.pow(2).sum()) / z1n.size(1)

def aug(x, noise_std=0.04, drop_p=0.12):
    x = x + noise_std * torch.randn_like(x)
    if drop_p > 0:
        m = (torch.rand_like(x) > drop_p).float()
        x = x * m
    return x

import importlib as _imp
import model_skeleton as _ms
_imp.reload(_ms)
from model_skeleton import MultiModalTiny

model = MultiModalTiny(dim=256, num_latents=6, num_layers=2, lambda_ortho=0.0).to(device)
model.train()
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=5e-4, weight_decay=1e-4)

probe = metrics_hooks.RedundancyProbe()

# fixed eval (same seed as before so the table is comparable)
def make_fixed_eval(N=256, seed=2025):
    g = np.random.default_rng(seed)
    imgs  = [Image.fromarray((g.random((224,224,3))*255).astype(np.uint8)) for _ in range(N)]
    texts = [f"fixed text {i}" for i in range(N)]
    auds  = [g.standard_normal(16000).astype(np.float32) for _ in range(N)]
    with torch.no_grad():
        img_512 = featurize_images(imgs); img_768 = img_adapter(img_512)
        txt_768 = featurize_text(texts);  aud_768 = featurize_audio(auds)
    return img_768, txt_768, aud_768

eval_img, eval_txt, eval_aud = make_fixed_eval(N=256, seed=2025)

# BEFORE
with torch.no_grad():
    _, zs0, _ = model(eval_img, eval_txt, eval_aud, compute_losses=False)
def show(tag, E):
    s = probe.all_stats(E.detach())
    return f"{tag:12s} | offdiag={s['offdiag_abs_mean']:.3f} | PR={s['participation_ratio']:.1f} | collapsed={s['num_collapsed_dims']} | topk_dup={s['avg_topk_dim_corr']:.3f}"
print("H1 BEFORE (fixed eval):")
print(show("img_shared", zs0["zsi"])); print(show("txt_shared", zs0["zst"])); print(show("aud_shared", zs0["zsa"]))

# train batches
def get_batch(N=256):
    imgs  = [Image.fromarray((np.random.rand(224,224,3)*255).astype(np.uint8)) for _ in range(N)]
    texts = [f"random text {i}" for i in range(N)]
    auds  = [np.random.randn(16000).astype(np.float32) for _ in range(N)]
    with torch.no_grad():
        img_512 = featurize_images(imgs); img_768 = img_adapter(img_512)
        txt_768 = featurize_text(texts);  aud_768 = featurize_audio(auds)
    return img_768, txt_768, aud_768

# weights – tuned to match the probe:
W_VAR = 60.0     # strong clamp to kill low-variance dims
W_COV = 5.0      # within-view correlation off-diagonal
W_BT  = 1.0      # modest cross-view alignment

steps = 250
for step in range(steps):
    img, txt, aud = get_batch(N=256)
    img1, img2 = aug(img), aug(img)
    txt1, txt2 = aug(txt), aug(txt)
    aud1, aud2 = aug(aud, 0.05, 0.15), aug(aud, 0.05, 0.15)

    # forward
    _, zA, _ = model(img1, txt1, aud1, compute_losses=False)
    _, zB, _ = model(img2, txt2, aud2, compute_losses=False)

    # per-modality objective (shared heads)
    def per_mod(z1, z2):
        return (
            W_BT * barlow_twins(z1, z2, lambd=1e-2) +
            W_COV * (corr_offdiag(z1) + corr_offdiag(z2)) +
            W_VAR * (variance_term(z1, gamma=1.0) + variance_term(z2, gamma=1.0))
        )

    L_img = per_mod(zA["zsi"], zB["zsi"])
    L_txt = per_mod(zA["zst"], zB["zst"])
    L_aud = per_mod(zA["zsa"], zB["zsa"])
    loss = L_img + L_txt + L_aud

    opt.zero_grad(); loss.backward(); opt.step()
    if step % 20 == 0:
        print(f"step {step:03d} | L_img={L_img.item():.3f} | L_txt={L_txt.item():.3f} | L_aud={L_aud.item():.3f}")

# AFTER (same fixed eval)
with torch.no_grad():
    _, zs1, _ = model(eval_img, eval_txt, eval_aud, compute_losses=False)

print("\nH1 AFTER (fixed eval):")
print(show("img_shared", zs1["zsi"]))
print(show("txt_shared", zs1["zst"]))
print(show("aud_shared", zs1["zsa"]))


H1 BEFORE (fixed eval):
img_shared   | offdiag=0.240 | PR=8.1 | collapsed=256 | topk_dup=0.608
txt_shared   | offdiag=0.191 | PR=13.7 | collapsed=256 | topk_dup=0.546
aud_shared   | offdiag=0.311 | PR=5.4 | collapsed=256 | topk_dup=0.746
step 000 | L_img=49.919 | L_txt=44.717 | L_aud=44.462
step 020 | L_img=1.493 | L_txt=1.194 | L_aud=1.208
step 040 | L_img=1.160 | L_txt=0.938 | L_aud=0.926
step 060 | L_img=1.134 | L_txt=0.840 | L_aud=0.862
step 080 | L_img=1.081 | L_txt=0.763 | L_aud=0.844
step 100 | L_img=1.076 | L_txt=0.713 | L_aud=0.806
step 120 | L_img=1.044 | L_txt=0.666 | L_aud=0.752
step 140 | L_img=1.025 | L_txt=0.631 | L_aud=0.744
step 160 | L_img=1.032 | L_txt=0.603 | L_aud=0.716
step 180 | L_img=1.016 | L_txt=0.576 | L_aud=0.660
step 200 | L_img=0.984 | L_txt=0.548 | L_aud=0.667
step 220 | L_img=0.979 | L_txt=0.534 | L_aud=0.622
step 240 | L_img=0.968 | L_txt=0.515 | L_aud=0.593

H1 AFTER (fixed eval):
img_shared   | offdiag=0.356 | PR=5.1 | collapsed=185 | topk_dup=0.782
t

In [35]:
# H3 with forward+backward timing (real encoders)

import time, torch, importlib
metrics_hooks = importlib.import_module("metrics_hooks")
from model_skeleton import MultiModalTiny

imgB, txtB, audB = make_batch(N=128)
yB = torch.randint(0,2,(imgB.size(0),), device=device)
criterion = nn.CrossEntropyLoss()

def bench(tokens=(4,6,8), layers=2, dim=256):
    for t in tokens:
        model = MultiModalTiny(dim=dim, num_latents=t, num_layers=layers, lambda_ortho=0.0).to(device)
        opt = torch.optim.SGD([p for p in model.parameters() if p.requires_grad], lr=1e-3)
        torch.cuda.reset_peak_memory_stats() if torch.cuda.is_available() else None
        start = time.time()
        logits, _, _ = model(imgB, txtB, audB, compute_losses=False)
        loss = criterion(logits, yB)
        opt.zero_grad(); loss.backward(); opt.step()
        vram_mb, iter_ms = metrics_hooks.log_vram_and_step_time(start)
        metric_val = float(torch.softmax(logits, dim=1).max(dim=1).values.mean().item())
        params_m = sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6
        print(f"MBT{t:>2} | VRAM {vram_mb:7.1f} MB | {iter_ms:7.1f} ms | proxy {metric_val:.3f} | params {params_m:.2f} M")

bench(tokens=(4,6,8), layers=2, dim=256)


MBT 4 | VRAM  1574.3 MB |     8.7 ms | proxy 0.580 | params 4.54 M
MBT 6 | VRAM  1605.5 MB |    10.1 ms | proxy 0.555 | params 4.54 M
MBT 8 | VRAM  1614.0 MB |     8.9 ms | proxy 0.532 | params 4.54 M


In [36]:
# Orthogonality trend on real features across a few iterations

import importlib, torch, numpy as np
from PIL import Image
metrics_hooks = importlib.import_module("metrics_hooks")
from model_skeleton import MultiModalTiny

device = "cuda" if torch.cuda.is_available() else "cpu"
model = MultiModalTiny(dim=256, num_latents=6, num_layers=2, lambda_ortho=0.02).to(device)
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3)
criterion = torch.nn.CrossEntropyLoss()

def get_batch(N=128):
    imgs  = [Image.fromarray((np.random.rand(224,224,3)*255).astype(np.uint8)) for _ in range(N)]
    texts = [f"random text {i}" for i in range(N)]
    auds  = [np.random.randn(16000).astype(np.float32) for _ in range(N)]
    with torch.no_grad():
        img_512 = featurize_images(imgs); img_768 = img_adapter(img_512)
        txt_768 = featurize_text(texts);  aud_768 = featurize_audio(auds)
    y = torch.randint(0,2,(N,), device=device)
    return img_768, txt_768, aud_768, y

for step in range(10):
    img, txt, aud, y = get_batch(N=128)
    logits, zs, losses = model(img, txt, aud, compute_losses=True)
    L_task, L_ortho = criterion(logits, y), losses["L_ortho"]
    loss = L_task + 0.02 * L_ortho
    opt.zero_grad(); loss.backward(); opt.step()

    sp = metrics_hooks.cross_cov_fro(zs["zsi"].detach(), zs["zpi"].detach())
    if step % 2 == 0:
        print(f"step {step} | L_task={L_task.item():.3f} | L_ortho={L_ortho.item():.3f} | cross_cov_fro(img shared↔private)={sp['cross_cov_fro']:.3f}")


step 0 | L_task=0.744 | L_ortho=199.725 | cross_cov_fro(img shared↔private)=53.512
step 2 | L_task=0.694 | L_ortho=159.978 | cross_cov_fro(img shared↔private)=51.934
step 4 | L_task=0.918 | L_ortho=114.607 | cross_cov_fro(img shared↔private)=46.332
step 6 | L_task=0.982 | L_ortho=73.769 | cross_cov_fro(img shared↔private)=36.874
step 8 | L_task=0.837 | L_ortho=59.465 | cross_cov_fro(img shared↔private)=33.562
